In [ ]:
from pathlib import Path
import itertools
import math
import re
import warnings

import numpy as np
import pandas as pd
from IPython.display import display, Markdown
from scipy.stats import ks_2samp
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import mutual_info_classif
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder, OneHotEncoder

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 180)

In [ ]:
DATASETS = {
    "embedding_dataset": Path("/content/data_lake_370records_5 (1).csv"),
    "cicids2017_dataset": Path("/content/cicids2017_leakage_safe (1).csv"),
}

TARGET_OVERRIDES = {
    "embedding_dataset": "label",
    "cicids2017_dataset": "label",
}

RANDOM_STATE = 42
OUTPUT_DIR = Path("audit_outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

In [ ]:
def normalized_name(value):
    return re.sub(r"[^a-z0-9]+", "_", str(value).strip().lower()).strip("_")


def read_csv_safely(path):
    attempts = [
        {"low_memory": False},
        {"sep": ";", "low_memory": False},
        {"sep": "\t", "low_memory": False},
    ]
    last_error = None
    for options in attempts:
        try:
            frame = pd.read_csv(path, **options)
            if frame.shape[1] > 1:
                return frame
        except Exception as exc:
            last_error = exc
    if last_error is not None:
        raise last_error
    raise ValueError(f"Unable to read {path}")


def infer_target(frame, configured_target=None):
    if configured_target is not None:
        if configured_target not in frame.columns:
            raise KeyError(f"Configured target column not found: {configured_target}")
        return configured_target
    candidates = [
        "label", "target", "class", "class_label", "attack", "category",
        "y", "outcome", "binary_label", "target_binary"
    ]
    lookup = {normalized_name(column): column for column in frame.columns}
    for candidate in candidates:
        if candidate in lookup:
            return lookup[candidate]
    low_cardinality = [
        column for column in frame.columns
        if 2 <= frame[column].nunique(dropna=False) <= min(30, max(2, len(frame) // 10))
    ]
    if not low_cardinality:
        raise ValueError("No target column could be inferred")
    return min(low_cardinality, key=lambda column: frame[column].nunique(dropna=False))


def infer_split_column(frame, target_column):
    split_tokens = {"split", "set", "partition", "fold", "dataset_split", "data_split"}
    for column in frame.columns:
        if column == target_column:
            continue
        name = normalized_name(column)
        values = frame[column].dropna().astype(str).str.lower().unique().tolist()
        if name in split_tokens:
            return column
        if any(token in name for token in ["split", "partition", "fold"]):
            if 2 <= len(values) <= 20:
                return column
    return None


def identifier_columns(frame, target_column, split_column):
    records = []
    tokens = {"id", "index", "uuid", "guid", "record", "row", "flow_id", "session_id", "sample_id"}
    for column in frame.columns:
        if column in {target_column, split_column}:
            continue
        series = frame[column]
        unique_ratio = series.nunique(dropna=False) / max(len(frame), 1)
        name = normalized_name(column)
        object_like = pd.api.types.is_object_dtype(series) or pd.api.types.is_string_dtype(series)
        integer_like = pd.api.types.is_integer_dtype(series)
        name_flag = any(token in name.split("_") for token in tokens) or name.endswith("_id")
        sequence_flag = False
        if integer_like and unique_ratio >= 0.98:
            values = pd.to_numeric(series, errors="coerce").dropna()
            sequence_flag = values.is_monotonic_increasing or values.is_monotonic_decreasing
        uniqueness_flag = object_like and unique_ratio >= 0.98
        if name_flag or sequence_flag or uniqueness_flag:
            records.append({
                "column": column,
                "unique_ratio": round(unique_ratio, 6),
                "name_based_flag": bool(name_flag),
                "near_unique_flag": bool(uniqueness_flag or sequence_flag),
            })
    return pd.DataFrame(records, columns=["column", "unique_ratio", "name_based_flag", "near_unique_flag"])


def quality_profile(frame, target_column, split_column, id_columns):
    rows = []
    for column in frame.columns:
        series = frame[column]
        non_null = series.dropna()
        unique_count = series.nunique(dropna=False)
        missing_count = int(series.isna().sum())
        missing_ratio = missing_count / max(len(frame), 1)
        top_frequency = series.value_counts(dropna=False).iloc[0] / max(len(frame), 1)
        numeric = pd.api.types.is_numeric_dtype(series)
        finite_count = np.nan
        infinity_count = 0
        zero_ratio = np.nan
        outlier_ratio = np.nan
        if numeric:
            values = pd.to_numeric(series, errors="coerce").to_numpy(dtype=float)
            finite = np.isfinite(values)
            finite_count = int(finite.sum())
            infinity_count = int((~finite & ~np.isnan(values)).sum())
            valid = values[finite]
            if valid.size:
                zero_ratio = float(np.mean(valid == 0))
                q1, q3 = np.quantile(valid, [0.25, 0.75])
                iqr = q3 - q1
                if iqr > 0:
                    outlier_ratio = float(np.mean((valid < q1 - 1.5 * iqr) | (valid > q3 + 1.5 * iqr)))
        rows.append({
            "column": column,
            "dtype": str(series.dtype),
            "missing_count": missing_count,
            "missing_ratio": round(missing_ratio, 6),
            "unique_count": int(unique_count),
            "unique_ratio": round(unique_count / max(len(frame), 1), 6),
            "top_value_ratio": round(float(top_frequency), 6),
            "constant": bool(unique_count <= 1),
            "quasi_constant": bool(top_frequency >= 0.995 and unique_count > 1),
            "numeric": bool(numeric),
            "finite_count": finite_count,
            "infinity_count": int(infinity_count),
            "zero_ratio": np.nan if pd.isna(zero_ratio) else round(zero_ratio, 6),
            "iqr_outlier_ratio": np.nan if pd.isna(outlier_ratio) else round(outlier_ratio, 6),
            "is_target": column == target_column,
            "is_split": column == split_column,
            "is_identifier": column in id_columns,
        })
    return pd.DataFrame(rows)


def exact_duplicate_columns(frame):
    representatives = []
    records = []
    for column in frame.columns:
        matched = None
        for representative in representatives:
            if frame[column].equals(frame[representative]):
                matched = representative
                break
        if matched is None:
            representatives.append(column)
        else:
            records.append({
                "representative_column": matched,
                "duplicate_column": column,
            })
    return pd.DataFrame(records, columns=["representative_column", "duplicate_column"])


def numeric_redundancy(frame, excluded_columns):
    numeric_columns = [
        column for column in frame.columns
        if column not in excluded_columns and pd.api.types.is_numeric_dtype(frame[column])
    ]
    if len(numeric_columns) < 2:
        return pd.DataFrame(columns=["feature_a", "feature_b", "pearson_correlation"])
    correlation = frame[numeric_columns].replace([np.inf, -np.inf], np.nan).corr()
    records = []
    for left_index, right_index in itertools.combinations(range(len(numeric_columns)), 2):
        left = numeric_columns[left_index]
        right = numeric_columns[right_index]
        value = correlation.iloc[left_index, right_index]
        if pd.notna(value) and abs(value) >= 0.995:
            records.append({
                "feature_a": left,
                "feature_b": right,
                "pearson_correlation": round(float(value), 8),
            })
    return pd.DataFrame(records).sort_values("pearson_correlation", key=lambda series: series.abs(), ascending=False) if records else pd.DataFrame(columns=["feature_a", "feature_b", "pearson_correlation"])


def categorical_leakage_audit(frame, target_column, excluded_columns):
    target = frame[target_column].astype("string").fillna("<MISSING_TARGET>")
    target_classes = target.nunique(dropna=False)
    records = []
    for column in frame.columns:
        if column in excluded_columns or column == target_column:
            continue
        series = frame[column]
        unique_count = series.nunique(dropna=False)
        low_cardinality = unique_count <= max(100, target_classes * 5)
        categorical_like = (
            pd.api.types.is_object_dtype(series)
            or pd.api.types.is_string_dtype(series)
            or pd.api.types.is_bool_dtype(series)
            or low_cardinality
        )
        if not categorical_like:
            continue
        feature = series.astype("string").fillna("<MISSING>")
        table = pd.crosstab(feature, target, dropna=False)
        purity = table.max(axis=1).sum() / max(len(frame), 1)
        target_to_feature = pd.crosstab(target, feature, dropna=False)
        determinism = target_to_feature.max(axis=1).sum() / max(len(frame), 1)
        feature_ratio = unique_count / max(len(frame), 1)
        direct_proxy = purity >= 0.995 and feature_ratio <= 0.5 and unique_count > 1
        target_derived = determinism >= 1 - 1e-12 and unique_count > 1
        near_target_derived = determinism >= 0.995 and not target_derived and unique_count > 1
        records.append({
            "column": column,
            "unique_count": int(unique_count),
            "unique_ratio": round(feature_ratio, 6),
            "feature_to_target_purity": round(float(purity), 6),
            "target_to_feature_determinism": round(float(determinism), 6),
            "direct_target_proxy_candidate": bool(direct_proxy),
            "target_derived_feature_candidate": bool(target_derived),
            "near_target_derived_association": bool(near_target_derived),
        })
    result = pd.DataFrame(records)
    if result.empty:
        return pd.DataFrame(columns=[
            "column", "unique_count", "unique_ratio", "feature_to_target_purity",
            "target_to_feature_determinism", "direct_target_proxy_candidate",
            "target_derived_feature_candidate", "near_target_derived_association"
        ])
    return result.sort_values(
        [
            "target_derived_feature_candidate",
            "direct_target_proxy_candidate",
            "near_target_derived_association",
            "feature_to_target_purity",
        ],
        ascending=[False, False, False, False]
    ).reset_index(drop=True)


def numeric_signal_audit(frame, target_column, excluded_columns):
    numeric_columns = [
        column for column in frame.columns
        if column not in excluded_columns and column != target_column and pd.api.types.is_numeric_dtype(frame[column])
    ]
    if not numeric_columns:
        return pd.DataFrame(columns=["column", "mutual_information"])
    matrix = frame[numeric_columns].replace([np.inf, -np.inf], np.nan).copy()
    matrix = matrix.fillna(matrix.median(numeric_only=True)).fillna(0)
    encoder = LabelEncoder()
    target = encoder.fit_transform(frame[target_column].astype("string").fillna("<MISSING_TARGET>"))
    discrete_mask = np.array([matrix[column].nunique(dropna=False) <= 20 for column in numeric_columns])
    scores = mutual_info_classif(
        matrix.to_numpy(),
        target,
        discrete_features=discrete_mask,
        random_state=RANDOM_STATE,
    )
    result = pd.DataFrame({
        "column": numeric_columns,
        "mutual_information": scores,
    })
    return result.sort_values("mutual_information", ascending=False).reset_index(drop=True)


def feature_fingerprints(frame, columns):
    if not columns:
        return pd.Series(np.arange(len(frame)), index=frame.index, dtype="uint64")
    normalized = frame[columns].copy()
    for column in normalized.columns:
        if pd.api.types.is_numeric_dtype(normalized[column]):
            normalized[column] = normalized[column].replace([np.inf, -np.inf], np.nan)
    return pd.util.hash_pandas_object(normalized, index=False)


def duplicate_observation_audit(frame, target_column, fingerprint):
    temporary = pd.DataFrame({
        "fingerprint": fingerprint.to_numpy(),
        "target": frame[target_column].astype("string").fillna("<MISSING_TARGET>").to_numpy(),
    })
    grouped = temporary.groupby("fingerprint", dropna=False)
    sizes = grouped.size()
    label_counts = grouped["target"].nunique(dropna=False)
    duplicate_groups = sizes[sizes > 1]
    conflicting_groups = label_counts[label_counts > 1]
    return pd.DataFrame([{
        "rows": len(frame),
        "duplicate_rows_beyond_first": int((duplicate_groups - 1).sum()),
        "duplicate_group_count": int(len(duplicate_groups)),
        "conflicting_label_group_count": int(len(conflicting_groups)),
        "conflicting_label_rows": int(sizes.loc[conflicting_groups.index].sum()) if len(conflicting_groups) else 0,
    }])


def split_overlap_audit(frame, split_column, target_column, fingerprint):
    columns = [
        "split_a", "split_b", "rows_a", "rows_b", "shared_unique_fingerprints",
        "rows_in_a_matching_b", "rows_in_b_matching_a", "overlap_rate_a", "overlap_rate_b"
    ]
    if split_column is None:
        return pd.DataFrame(columns=columns)
    split = frame[split_column].astype("string").fillna("<MISSING_SPLIT>")
    unique_splits = sorted(split.unique().tolist())
    records = []
    for left, right in itertools.combinations(unique_splits, 2):
        left_mask = split == left
        right_mask = split == right
        left_fp = fingerprint[left_mask]
        right_fp = fingerprint[right_mask]
        left_set = set(left_fp.tolist())
        right_set = set(right_fp.tolist())
        shared = left_set.intersection(right_set)
        left_matches = int(left_fp.isin(shared).sum())
        right_matches = int(right_fp.isin(shared).sum())
        records.append({
            "split_a": left,
            "split_b": right,
            "rows_a": int(left_mask.sum()),
            "rows_b": int(right_mask.sum()),
            "shared_unique_fingerprints": int(len(shared)),
            "rows_in_a_matching_b": left_matches,
            "rows_in_b_matching_a": right_matches,
            "overlap_rate_a": round(left_matches / max(int(left_mask.sum()), 1), 6),
            "overlap_rate_b": round(right_matches / max(int(right_mask.sum()), 1), 6),
        })
    return pd.DataFrame(records, columns=columns)


def split_distribution_audit(frame, split_column, target_column, feature_columns):
    columns = ["feature", "split_a", "split_b", "ks_statistic", "ks_p_value"]
    if split_column is None:
        return pd.DataFrame(columns=columns)
    split = frame[split_column].astype("string").fillna("<MISSING_SPLIT>")
    unique_splits = sorted(split.unique().tolist())
    numeric_columns = [
        column for column in feature_columns
        if pd.api.types.is_numeric_dtype(frame[column])
    ]
    records = []
    for left, right in itertools.combinations(unique_splits, 2):
        left_mask = split == left
        right_mask = split == right
        for column in numeric_columns:
            left_values = pd.to_numeric(frame.loc[left_mask, column], errors="coerce").replace([np.inf, -np.inf], np.nan).dropna()
            right_values = pd.to_numeric(frame.loc[right_mask, column], errors="coerce").replace([np.inf, -np.inf], np.nan).dropna()
            if len(left_values) < 5 or len(right_values) < 5:
                continue
            statistic, p_value = ks_2samp(left_values, right_values)
            records.append({
                "feature": column,
                "split_a": left,
                "split_b": right,
                "ks_statistic": float(statistic),
                "ks_p_value": float(p_value),
            })
    if not records:
        return pd.DataFrame(columns=columns)
    return pd.DataFrame(records).sort_values("ks_statistic", ascending=False).reset_index(drop=True)


def select_model_split(frame, target_column, split_column):
    target = frame[target_column].astype("string").fillna("<MISSING_TARGET>")
    if split_column is not None:
        split = frame[split_column].astype("string").str.lower()
        train_mask = split.isin({"train", "training"})
        test_mask = split.isin({"test", "testing"})
        if train_mask.sum() > 0 and test_mask.sum() > 0:
            return np.flatnonzero(train_mask.to_numpy()), np.flatnonzero(test_mask.to_numpy()), "declared_train_test"
    indices = np.arange(len(frame))
    counts = target.value_counts()
    stratify = target if counts.min() >= 2 else None
    train_index, test_index = train_test_split(
        indices,
        test_size=0.25,
        random_state=RANDOM_STATE,
        stratify=stratify,
    )
    return train_index, test_index, "generated_stratified_holdout"


def make_pipeline(frame, columns):
    numeric_columns = [column for column in columns if pd.api.types.is_numeric_dtype(frame[column])]
    categorical_columns = [column for column in columns if column not in numeric_columns]
    transformers = []
    if numeric_columns:
        transformers.append((
            "numeric",
            Pipeline([("imputer", SimpleImputer(strategy="median"))]),
            numeric_columns,
        ))
    if categorical_columns:
        transformers.append((
            "categorical",
            Pipeline([
                ("imputer", SimpleImputer(strategy="most_frequent")),
                ("encoder", OneHotEncoder(handle_unknown="ignore")),
            ]),
            categorical_columns,
        ))
    preprocessor = ColumnTransformer(transformers=transformers, remainder="drop")
    model = RandomForestClassifier(
        n_estimators=180,
        random_state=RANDOM_STATE,
        n_jobs=-1,
        class_weight="balanced_subsample",
        min_samples_leaf=1,
    )
    return Pipeline([("preprocessor", preprocessor), ("model", model)])


def evaluate_feature_sets(frame, target_column, split_column, full_features, safe_features):
    train_index, test_index, split_mode = select_model_split(frame, target_column, split_column)
    y = frame[target_column].astype("string").fillna("<MISSING_TARGET>")
    results = []
    for name, features in [("full_candidate_features", full_features), ("safe_features", safe_features)]:
        if not features:
            continue
        pipeline = make_pipeline(frame, features)
        pipeline.fit(frame.iloc[train_index][features], y.iloc[train_index])
        prediction = pipeline.predict(frame.iloc[test_index][features])
        results.append({
            "feature_set": name,
            "evaluation_split": split_mode,
            "feature_count": len(features),
            "train_rows": len(train_index),
            "test_rows": len(test_index),
            "accuracy": round(float(accuracy_score(y.iloc[test_index], prediction)), 6),
            "balanced_accuracy": round(float(balanced_accuracy_score(y.iloc[test_index], prediction)), 6),
            "macro_f1": round(float(f1_score(y.iloc[test_index], prediction, average="macro", zero_division=0)), 6),
        })
    return pd.DataFrame(results)


def make_safe_feature_list(frame, target_column, split_column, id_columns, profile, duplicates, leakage):
    excluded = {target_column}
    if split_column is not None:
        excluded.add(split_column)
    excluded.update(id_columns)
    excluded.update(profile.loc[profile["constant"], "column"].tolist())
    if not duplicates.empty:
        excluded.update(duplicates["duplicate_column"].tolist())
    if not leakage.empty:
        suspicious = leakage.loc[
            leakage["direct_target_proxy_candidate"] | leakage["target_derived_feature_candidate"],
            "column"
        ].tolist()
        excluded.update(suspicious)
    full_features = [
        column for column in frame.columns
        if column not in {target_column, split_column} and column not in id_columns
    ]
    safe_features = [column for column in full_features if column not in excluded]
    return full_features, safe_features, sorted(excluded)


def risk_summary(profile, ids, duplicates, leakage, overlap):
    reasons = []
    if not leakage.empty:
        derived = leakage["target_derived_feature_candidate"].sum()
        direct = leakage["direct_target_proxy_candidate"].sum()
        if derived:
            reasons.append(f"{int(derived)} target-derived feature candidate(s)")
        if direct:
            reasons.append(f"{int(direct)} direct target-proxy candidate(s)")
    if not overlap.empty:
        maximum_overlap = overlap[["overlap_rate_a", "overlap_rate_b"]].max().max()
        if maximum_overlap > 0:
            reasons.append(f"cross-split feature overlap up to {maximum_overlap:.2%}")
    if len(ids):
        reasons.append(f"{len(ids)} identifier-like column(s)")
    if len(duplicates):
        reasons.append(f"{len(duplicates)} exact duplicate-column relationship(s)")
    constants = int(profile["constant"].sum())
    if constants:
        reasons.append(f"{constants} constant column(s)")
    if any("target-derived" in reason or "direct target" in reason or "cross-split" in reason for reason in reasons):
        level = "HIGH"
    elif reasons:
        level = "MODERATE"
    else:
        level = "LOW"
    return level, reasons

In [ ]:
audit_results = {}

for dataset_name, dataset_path in DATASETS.items():
    if not dataset_path.exists():
        raise FileNotFoundError(f"Dataset not found: {dataset_path}")

    frame = read_csv_safely(dataset_path)
    target_column = infer_target(frame, TARGET_OVERRIDES.get(dataset_name))
    split_column = infer_split_column(frame, target_column)

    id_report = identifier_columns(frame, target_column, split_column)
    id_columns = id_report["column"].tolist()

    profile = quality_profile(frame, target_column, split_column, id_columns)
    duplicates = exact_duplicate_columns(frame)
    leakage = categorical_leakage_audit(
        frame,
        target_column,
        excluded_columns=set(id_columns + ([split_column] if split_column is not None else [])),
    )
    numeric_signal = numeric_signal_audit(
        frame,
        target_column,
        excluded_columns=set(id_columns + ([split_column] if split_column is not None else [])),
    )
    full_features, safe_features, safe_exclusions = make_safe_feature_list(
        frame,
        target_column,
        split_column,
        id_columns,
        profile,
        duplicates,
        leakage,
    )
    fingerprint = feature_fingerprints(frame, full_features)
    duplicate_rows = duplicate_observation_audit(frame, target_column, fingerprint)
    overlap = split_overlap_audit(frame, split_column, target_column, fingerprint)
    redundancy = numeric_redundancy(frame, {target_column, split_column})
    split_drift = split_distribution_audit(frame, split_column, target_column, full_features)
    evaluation = evaluate_feature_sets(
        frame,
        target_column,
        split_column,
        full_features,
        safe_features,
    )
    risk_level, risk_reasons = risk_summary(profile, id_report, duplicates, leakage, overlap)

    summary = pd.DataFrame([{
        "dataset": dataset_name,
        "source_file": dataset_path.name,
        "rows": len(frame),
        "columns": len(frame.columns),
        "target_column": target_column,
        "target_classes": int(frame[target_column].nunique(dropna=False)),
        "split_column": split_column if split_column is not None else "None",
        "missing_cells": int(frame.isna().sum().sum()),
        "exact_duplicate_rows": int(frame.duplicated().sum()),
        "constant_columns": int(profile["constant"].sum()),
        "identifier_like_columns": len(id_report),
        "exact_duplicate_column_relationships": len(duplicates),
        "duplicate_rows_using_candidate_features": int(duplicate_rows.loc[0, "duplicate_rows_beyond_first"]),
        "conflicting_label_duplicate_groups": int(duplicate_rows.loc[0, "conflicting_label_group_count"]),
        "leakage_risk_level": risk_level,
        "risk_reasons": "; ".join(risk_reasons) if risk_reasons else "No automated risk trigger",
    }])

    print(f"\n{'=' * 100}\n{dataset_name}\n{'=' * 100}")
    display(summary)
    display(Markdown("### Target Distribution"))
    display(frame[target_column].value_counts(dropna=False).rename_axis(target_column).reset_index(name="count"))

    if split_column is not None:
        display(Markdown("### Target Distribution by Declared Split"))
        display(pd.crosstab(frame[split_column], frame[target_column], margins=True))

    display(Markdown("### Data Quality Flags"))
    display(profile.loc[
        profile["constant"] | profile["quasi_constant"] | (profile["missing_ratio"] > 0) |
        (profile["infinity_count"] > 0) | (profile["is_identifier"]),
        ["column", "dtype", "missing_count", "unique_count", "top_value_ratio", "constant",
         "quasi_constant", "infinity_count", "zero_ratio", "iqr_outlier_ratio", "is_identifier"]
    ].sort_values(["constant", "is_identifier", "missing_count"], ascending=[False, False, False]))

    display(Markdown("### Identifier-Like Columns"))
    display(id_report)

    display(Markdown("### Exact Duplicate Columns"))
    display(duplicates.head(100))

    display(Markdown("### High-Correlation Numeric Feature Pairs"))
    display(redundancy.head(100))

    display(Markdown("### Potential Target Leakage and Target-Derived Fields"))
    display(leakage.head(100))

    display(Markdown("### Strongest Numeric Associations With the Target"))
    display(numeric_signal.head(25))

    display(Markdown("### Duplicate Observation Audit"))
    display(duplicate_rows)

    if split_column is not None:
        display(Markdown("### Cross-Split Feature Overlap"))
        display(overlap)
        display(Markdown("### Largest Numeric Distribution Differences Across Splits"))
        display(split_drift.head(30))

    display(Markdown("### Model Sensitivity to Suspicious Features"))
    display(evaluation)

    prefix = OUTPUT_DIR / dataset_name
    summary.to_csv(f"{prefix}_summary.csv", index=False)
    profile.to_csv(f"{prefix}_column_quality.csv", index=False)
    id_report.to_csv(f"{prefix}_identifier_columns.csv", index=False)
    duplicates.to_csv(f"{prefix}_duplicate_columns.csv", index=False)
    redundancy.to_csv(f"{prefix}_numeric_redundancy.csv", index=False)
    leakage.to_csv(f"{prefix}_leakage_candidates.csv", index=False)
    numeric_signal.to_csv(f"{prefix}_numeric_signal.csv", index=False)
    duplicate_rows.to_csv(f"{prefix}_duplicate_observations.csv", index=False)
    overlap.to_csv(f"{prefix}_cross_split_overlap.csv", index=False)
    split_drift.to_csv(f"{prefix}_split_distribution_drift.csv", index=False)
    evaluation.to_csv(f"{prefix}_model_sensitivity.csv", index=False)

    audit_results[dataset_name] = {
        "frame": frame,
        "summary": summary,
        "profile": profile,
        "id_report": id_report,
        "duplicates": duplicates,
        "redundancy": redundancy,
        "leakage": leakage,
        "numeric_signal": numeric_signal,
        "duplicate_rows": duplicate_rows,
        "overlap": overlap,
        "split_drift": split_drift,
        "evaluation": evaluation,
        "safe_exclusions": safe_exclusions,
        "full_features": full_features,
        "safe_features": safe_features,
    }


embedding_dataset


,dataset,source_file,rows,columns,target_column,target_classes,split_column,missing_cells,exact_duplicate_rows,constant_columns,identifier_like_columns,exact_duplicate_column_relationships,duplicate_rows_using_candidate_features,conflicting_label_duplicate_groups,leakage_risk_level,risk_reasons
0,embedding_dataset,data_lake_370records_5 (1).csv,370,130,label,10,None,0,0,4,0,29,0,0,MODERATE,29 exact duplicate-column relationship(s); 4 c...


### Target Distribution

,label,count
0,Normal,100
1,Exploit,30
2,Worm,30
3,DoS,30
4,Shellcode,30
5,Probe,30
6,Reconnaissance,30
7,U2R,30
8,R2L,30
9,Backdoor,30


### Data Quality Flags

...

,split_a,split_b,rows_a,rows_b,shared_unique_fingerprints,rows_in_a_matching_b,rows_in_b_matching_a,overlap_rate_a,overlap_rate_b
0,test,train,253,1265,22,24,32,0.094862,0.025296
1,test,validation,253,253,4,4,4,0.015810,0.015810
2,train,validation,1265,253,23,31,25,0.024506,0.098814


### Largest Numeric Distribution Differences Across Splits

,feature,split_a,split_b,ks_statistic,ks_p_value
0,Init_Win_bytes_backward,test,validation,0.114625,0.071955
1,act_data_pkt_fwd,test,train,0.084585,0.094214
2,Subflow Fwd Bytes,test,validation,0.083004,0.348626
3,Init_Win_bytes_forward,test,validation,0.083004,0.348626
4,Total Length of Fwd Packets,test,validation,0.083004,0.348626
5,Init_Win_bytes_backward,train,validation,0.081423,0.117750
6,Packet Length Mean,test,train,0.079842,0.131217
7,Flow IAT Max,test,validation,0.079051,0.408602
8,Avg Bwd Segment Size,test,train,0.079051,0.138406
9,Bwd Packet Length Mean,test,train,0.079051,0.138406


### Model Sensitivity to Suspicious Features

,feature_set,evaluation_split,feature_count,train_rows,test_rows,accuracy,balanced_accuracy,macro_f1
0,full_candidate_features,declared_train_test,78,1265,253,0.877470,0.829516,0.825407
1,safe_features,declared_train_test,57,1265,253,0.865613,0.818555,0.832282


In [ ]:
consolidated = pd.concat(
    [result["summary"] for result in audit_results.values()],
    ignore_index=True,
)
display(consolidated)

for dataset_name, result in audit_results.items():
    print(f"\n{dataset_name}")
    print(f"Candidate features: {len(result['full_features'])}")
    print(f"Safe features after exclusions: {len(result['safe_features'])}")
    print(f"Excluded columns: {', '.join(result['safe_exclusions']) if result['safe_exclusions'] else 'None'}")
    if not result["evaluation"].empty and len(result["evaluation"]) == 2:
        full_score = float(result["evaluation"].loc[
            result["evaluation"]["feature_set"] == "full_candidate_features", "balanced_accuracy"
        ].iloc[0])
        safe_score = float(result["evaluation"].loc[
            result["evaluation"]["feature_set"] == "safe_features", "balanced_accuracy"
        ].iloc[0])
        print(f"Balanced accuracy change after safe-feature filtering: {safe_score - full_score:+.6f}")

,dataset,source_file,rows,columns,target_column,target_classes,split_column,missing_cells,exact_duplicate_rows,constant_columns,identifier_like_columns,exact_duplicate_column_relationships,duplicate_rows_using_candidate_features,conflicting_label_duplicate_groups,leakage_risk_level,risk_reasons
0,embedding_dataset,data_lake_370records_5 (1).csv,370,130,label,10,None,0,0,4,0,29,0,0,MODERATE,29 exact duplicate-column relationship(s); 4 c...
1,cicids2017_dataset,cicids2017_leakage_safe (1).csv,1771,81,label,6,split,0,0,12,1,19,108,0,HIGH,1 target-derived feature candidate(s); cross-s...



embedding_dataset
Candidate features: 129
Safe features after exclusions: 99
Excluded columns: dim_11, dim_22, dim_28, dim_29, dim_30, dim_31, dim_32, dim_33, dim_34, dim_35, dim_36, dim_37, dim_38, dim_39, dim_40, dim_41, dim_42, dim_43, dim_44, dim_45, dim_46, dim_47, dim_48, dim_49, dim_50, dim_51, dim_52, dim_53, dim_54, dim_55, label
Balanced accuracy change after safe-feature filtering: +0.000000

cicids2017_dataset
Candidate features: 78
Safe features after exclusions: 57
Excluded columns: Avg Bwd Segment Size, Avg Fwd Segment Size, Bwd Avg Bulk Rate, Bwd Avg Bytes/Bulk, Bwd Avg Packets/Bulk, Bwd PSH Flags, Bwd URG Flags, CWE Flag Count, ECE Flag Count, Fwd Avg Bulk Rate, Fwd Avg Bytes/Bulk, Fwd Avg Packets/Bulk, Fwd Header Length.1, Fwd URG Flags, RST Flag Count, SYN Flag Count, Subflow Bwd Bytes, Subflow Bwd Packets, Subflow Fwd Bytes, Subflow Fwd Packets, label, row_id, split, target_binary
Balanced accuracy change after safe-feature filtering: -0.010961
